# KV Cache Trace Workload Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tteon/kcc2026-tutorial/blob/main/kvcache_trace_workload_analysis.ipynb)

> On Colab, upload your trace as `trace.jsonl` (or paste it into `RAW_JSONL` in the config cell) before running.

This notebook analyzes JSON Lines traces where each row looks like:

```json
{"timestamp": 0, "input_length": 6758, "output_length": 500, "hash_ids": [0, 1, 2, ...]}
```

Analysis goals:

1. Request-level workload statistics: input/output length, total tokens, p50/p99, arrival gaps.
2. KV-cache-aware statistics: block reuse, prefix sharing, working set, cache pressure.
3. Lifespan analysis: first/last use, reuse distance, estimated live KV memory, LRU sensitivity.

In [1]:
# Core libraries only: pandas/numpy/matplotlib are enough for this trace-style analysis.
# No seaborn is used so that the plots stay simple and portable.

from __future__ import annotations

import json
from pathlib import Path
from collections import OrderedDict
from typing import Iterable, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Configuration

Adjust this cell first.

Important note: many KV traces use a special root/sentinel hash such as `0`. If `0` is not a real cache block, set `EXCLUDE_HASH_IDS = {0}`. Keeping such a sentinel can artificially inflate cache reuse statistics.

In [2]:
# Path to your JSONL file.
# Example: put your trace in the same folder as this notebook as `trace.jsonl`.
TRACE_PATH = Path("trace.jsonl")

# Alternative: paste a small JSONL sample here if you want to test quickly.
# Keep it empty when loading from TRACE_PATH.
RAW_JSONL = ""   # paste JSONL text here if needed

# If hash_id 0 is a sentinel/root/shared marker rather than a real KV block, use {0}.
# If it is a real KV block, keep set().
EXCLUDE_HASH_IDS = set()       # e.g. {0}

# Timestamp interpretation for labels only. The raw values are always preserved.
# Your sample uses 0 and 3000, so "ms" may be reasonable if 3000 means 3 seconds.
TIMESTAMP_UNIT = "ms"          # one of: "raw", "ms", "s"

# KV block size is usually engine-dependent. Set this to your serving engine's block size if known.
BLOCK_SIZE_TOKENS = 16

# Model config for approximate KV memory.
# Formula per token: n_layers * 2(K,V) * n_kv_heads * head_dim * dtype_bytes
# Example values below are placeholders; replace them with your model's real config.
KV_MODEL_CONFIG = {
    "n_layers": 32,
    "n_kv_heads": 8,
    "head_dim": 128,
    "dtype_bytes": 2,     # fp16/bf16 = 2, fp8/int8-like = 1
}

# LRU capacities are in number of KV blocks, not bytes.
LRU_CAPACITIES = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]

# Sliding-window working set sizes in request count.
WORKING_SET_WINDOWS = [10, 50, 100, 500]

## 2. Load and normalize the trace

This loader accepts either a `.jsonl` file or the pasted `RAW_JSONL` text.

In [3]:
def _normalize_hash_id(x: Any) -> Any:
    """Normalize numeric-looking hash IDs to int; keep non-numeric IDs as strings."""
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float) and x.is_integer():
        return int(x)
    if isinstance(x, str):
        s = x.strip()
        try:
            return int(s)
        except ValueError:
            return s
    return x


def _normalize_hash_list(ids: Any, exclude_hash_ids: set[Any] | None = None) -> list[Any]:
    """Clean hash_ids into a list and optionally remove sentinel/root IDs."""
    exclude_hash_ids = {_normalize_hash_id(x) for x in (exclude_hash_ids or set())}
    if ids is None or (isinstance(ids, float) and np.isnan(ids)):
        return []
    if not isinstance(ids, list):
        raise TypeError(f"hash_ids must be a list, got: {type(ids)}")
    out = []
    for x in ids:
        x_norm = _normalize_hash_id(x)
        if x_norm not in exclude_hash_ids:
            out.append(x_norm)
    return out


def load_trace_jsonl(path: Path | str | None = None, raw_jsonl: str | None = None) -> pd.DataFrame:
    """Load the KV trace JSON Lines into a pandas DataFrame."""
    records = []

    if raw_jsonl and raw_jsonl.strip():
        lines = raw_jsonl.strip().splitlines()
        source_desc = "RAW_JSONL"
    else:
        path = Path(path or TRACE_PATH)
        if not path.exists():
            raise FileNotFoundError(
                f"Trace file not found: {path}. Put a JSONL file there or paste data into RAW_JSONL."
            )
        lines = path.read_text(encoding="utf-8").splitlines()
        source_desc = str(path)

    for line_no, line in enumerate(lines, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON at line {line_no} in {source_desc}: {e}") from e

    df = pd.DataFrame(records)

    required = {"timestamp", "input_length", "output_length", "hash_ids"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Preserve original row order for tie-breaking when timestamps are equal.
    df["original_row_id"] = np.arange(len(df))

    # Numeric conversion with explicit errors for cleaner debugging.
    for col in ["timestamp", "input_length", "output_length"]:
        df[col] = pd.to_numeric(df[col], errors="raise")

    # Sort by timestamp, then by original order. This gives a deterministic request order.
    df = df.sort_values(["timestamp", "original_row_id"], kind="mergesort").reset_index(drop=True)
    df["request_id"] = np.arange(len(df))

    # Keep both all hash IDs and filtered hash IDs.
    df["hash_ids_all"] = df["hash_ids"].apply(lambda x: _normalize_hash_list(x, exclude_hash_ids=set()))
    df["hash_ids"] = df["hash_ids"].apply(lambda x: _normalize_hash_list(x, exclude_hash_ids=EXCLUDE_HASH_IDS))

    # Request-level derived fields.
    df["input_tokens"] = df["input_length"]
    df["output_tokens"] = df["output_length"]
    df["total_tokens"] = df["input_length"] + df["output_length"]
    df["output_input_ratio"] = np.where(df["input_length"] > 0, df["output_length"] / df["input_length"], np.nan)

    df["num_hash_blocks"] = df["hash_ids"].apply(len)
    df["num_unique_hash_blocks"] = df["hash_ids"].apply(lambda x: len(set(x)))
    df["duplicate_blocks_in_request"] = df["num_hash_blocks"] - df["num_unique_hash_blocks"]

    # Timestamp helpers for plots.
    if TIMESTAMP_UNIT == "ms":
        df["time_for_plot"] = df["timestamp"] / 1000.0
        df.attrs["time_label"] = "time (s)"
    elif TIMESTAMP_UNIT == "s":
        df["time_for_plot"] = df["timestamp"]
        df.attrs["time_label"] = "time (s)"
    else:
        df["time_for_plot"] = df["timestamp"]
        df.attrs["time_label"] = "timestamp (raw)"

    # Inter-arrival gap. With batched timestamps, many gaps may be zero.
    df["arrival_gap_raw"] = df["timestamp"].diff()
    df["arrival_gap_for_plot"] = df["time_for_plot"].diff()

    return df


trace_df = load_trace_jsonl(TRACE_PATH, RAW_JSONL)
print(f"Loaded {len(trace_df):,} requests")
print(f"Hash IDs excluded from cache-aware stats: {EXCLUDE_HASH_IDS}")
display(trace_df.head())

Loaded 43,058 requests
Hash IDs excluded from cache-aware stats: set()


,chat_id,parent_chat_id,timestamp,input_length,output_length,type,turn,hash_ids,original_row_id,request_id,hash_ids_all,input_tokens,output_tokens,total_tokens,output_input_ratio,num_hash_blocks,num_unique_hash_blocks,duplicate_blocks_in_request,time_for_plot,arrival_gap_raw,arrival_gap_for_plot
0,0,-1,0.000,791,532,text,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",791,532,1323,0.672566,50,50,0,0.000000,NaN,NaN
1,1,-1,0.206,378,76,image,1,"[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6...",1,1,"[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6...",378,76,454,0.201058,24,24,0,0.000206,0.206,0.000206
2,2,-1,0.223,16744,540,search,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",16744,540,17284,0.032250,1047,1046,1,0.000223,0.017,0.000017
3,3,-1,0.718,358,54,image,1,"[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6...",3,3,"[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6...",358,54,412,0.150838,23,23,0,0.000718,0.495,0.000495
4,4,-1,1.073,756,211,text,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",4,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",756,211,967,0.279101,48,48,0,0.001073,0.355,0.000355


## 3. Basic request-level statistics

Start with standard workload statistics before doing cache-aware analysis.

In [4]:
PERCENTILES = [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]


def percentile_table(df: pd.DataFrame, columns: list[str], percentiles=PERCENTILES) -> pd.DataFrame:
    """Return count/mean/std/min/max and requested percentiles for selected columns."""
    rows = []
    for col in columns:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        q = s.quantile(percentiles)
        row = {
            "metric": col,
            "count": int(s.count()),
            "mean": s.mean(),
            "std": s.std(ddof=1),
            "min": s.min(),
            "max": s.max(),
        }
        row.update({f"p{int(p * 100):02d}": q.loc[p] for p in percentiles})
        rows.append(row)
    return pd.DataFrame(rows)


request_metrics = [
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "output_input_ratio",
    "num_hash_blocks",
    "num_unique_hash_blocks",
    "duplicate_blocks_in_request",
    "arrival_gap_raw",
]

request_stats = percentile_table(trace_df, request_metrics)
display(request_stats)

,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,input_tokens,43058,2331.337150,3332.166953,56.000000,89286.000000,56.000000,69.000000,75.000000,451.000000,1046.000000,2866.750000,6436.000000,8808.15000,14364.590000,89286.000000
1,output_tokens,43058,430.472595,351.486631,1.000000,8000.000000,1.000000,15.000000,51.000000,209.000000,375.500000,551.000000,808.000000,1032.00000,1641.430000,8000.000000
2,total_tokens,43058,2761.809745,3450.986101,70.000000,89968.000000,70.000000,145.000000,303.000000,797.000000,1422.000000,3448.000000,7049.000000,9459.30000,15204.160000,89968.000000
3,output_input_ratio,43058,0.886380,1.766696,0.000054,27.929577,0.000054,0.012607,0.036968,0.112875,0.283652,0.639671,2.591981,4.73913,8.772813,27.929577
4,num_hash_blocks,43058,146.174369,208.261759,4.000000,5581.000000,4.000000,5.000000,5.000000,29.000000,66.000000,180.000000,403.000000,551.00000,898.430000,5581.000000
5,num_unique_hash_blocks,43058,143.711970,193.609507,4.000000,3009.000000,4.000000,5.000000,5.000000,29.000000,66.000000,178.000000,396.000000,543.00000,883.000000,3009.000000
6,duplicate_blocks_in_request,43058,2.462400,37.708571,0.000000,2572.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,6.00000,35.000000,2572.000000
7,arrival_gap_raw,43057,0.167208,0.176739,0.000000,3.154000,0.000000,0.002000,0.008000,0.046000,0.113000,0.228000,0.387000,0.51400,0.815000,3.154000


In [5]:
# Aggregate by timestamp to inspect bursts/batches.

def count_unique_blocks_in_series(series: pd.Series) -> int:
    return len({b for ids in series for b in ids})


time_df = (
    trace_df.groupby("timestamp", as_index=False)
    .agg(
        time_for_plot=("time_for_plot", "first"),
        n_requests=("request_id", "count"),
        input_tokens=("input_tokens", "sum"),
        output_tokens=("output_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
        hash_block_refs=("num_hash_blocks", "sum"),
        unique_hash_blocks=("hash_ids", count_unique_blocks_in_series),
    )
)

time_df["tokens_per_request"] = time_df["total_tokens"] / time_df["n_requests"]
time_df["hash_block_refs_per_request"] = time_df["hash_block_refs"] / time_df["n_requests"]

display(time_df.head(20))
display(percentile_table(time_df, ["n_requests", "total_tokens", "hash_block_refs", "unique_hash_blocks"]))

,timestamp,time_for_plot,n_requests,input_tokens,output_tokens,total_tokens,hash_block_refs,unique_hash_blocks,tokens_per_request,hash_block_refs_per_request
0,0.000,0.000000,1,791,532,1323,50,50,1323.0,50.0
1,0.206,0.000206,1,378,76,454,24,24,454.0,24.0
2,0.223,0.000223,1,16744,540,17284,1047,1046,17284.0,1047.0
3,0.718,0.000718,1,358,54,412,23,23,412.0,23.0
4,1.073,0.001073,1,756,211,967,48,48,967.0,48.0
5,1.191,0.001191,1,805,347,1152,51,51,1152.0,51.0
6,1.506,0.001506,1,750,275,1025,47,47,1025.0,47.0
7,1.906,0.001906,1,3023,373,3396,189,186,3396.0,189.0
8,2.603,0.002603,1,494,277,771,31,31,771.0,31.0
9,3.297,0.003297,1,409,334,743,26,26,743.0,26.0


,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,n_requests,42919,1.003239,0.056818,1,2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.00,2.0
1,total_tokens,42919,2770.754305,3461.256394,70,89968,70.0,145.0,305.9,799.0,1426.0,3461.0,7078.2,9491.3,15221.82,89968.0
2,hash_block_refs,42919,146.647778,208.821372,4,5581,4.0,5.0,5.0,29.0,66.0,180.0,404.2,553.0,902.00,5581.0
3,unique_hash_blocks,42919,144.152357,194.134681,4,3009,4.0,5.0,5.0,29.0,66.0,179.0,397.0,544.0,885.00,3009.0


## 4. KV-cache-aware reuse statistics

Two interpretations are useful:

1. `sequential`: earlier rows at the same timestamp can warm the cache for later rows.
2. `batch`: rows with the same timestamp are treated as simultaneous, so same-timestamp rows do not warm the cache for each other.

For serving traces, `batch` is often more conservative when timestamp resolution is coarse.

In [6]:
def compute_history_reuse(df: pd.DataFrame, policy: str = "batch", ids_col: str = "hash_ids") -> pd.DataFrame:
    """
    Compute per-request reuse against blocks seen before this request.

    policy="sequential": process each row in order.
    policy="batch": requests with the same timestamp only see blocks from earlier timestamps.
    """
    if policy not in {"sequential", "batch"}:
        raise ValueError("policy must be either 'sequential' or 'batch'")

    rows = []
    seen: set[Any] = set()

    def summarize_request(row, seen_before: set[Any]) -> dict[str, Any]:
        ids = list(row[ids_col])
        unique_ids = set(ids)
        hit_refs = sum(1 for b in ids if b in seen_before)
        miss_refs = len(ids) - hit_refs
        hit_unique = sum(1 for b in unique_ids if b in seen_before)
        miss_unique = len(unique_ids) - hit_unique
        return {
            "request_id": row["request_id"],
            f"{policy}_history_hit_block_refs": hit_refs,
            f"{policy}_history_miss_block_refs": miss_refs,
            f"{policy}_history_hit_unique_blocks": hit_unique,
            f"{policy}_history_miss_unique_blocks": miss_unique,
            f"{policy}_history_hit_rate_refs": hit_refs / len(ids) if ids else np.nan,
            f"{policy}_history_hit_rate_unique": hit_unique / len(unique_ids) if unique_ids else np.nan,
        }

    if policy == "sequential":
        for _, row in df.iterrows():
            rows.append(summarize_request(row, seen))
            seen.update(row[ids_col])

    else:  # batch policy
        for _, g in df.groupby("timestamp", sort=True):
            # Every request in this timestamp group sees only blocks from earlier timestamps.
            batch_new_blocks = set()
            for _, row in g.iterrows():
                rows.append(summarize_request(row, seen))
                batch_new_blocks.update(row[ids_col])
            seen.update(batch_new_blocks)

    return pd.DataFrame(rows).sort_values("request_id").reset_index(drop=True)


seq_reuse = compute_history_reuse(trace_df, policy="sequential")
batch_reuse = compute_history_reuse(trace_df, policy="batch")

trace_df = trace_df.merge(seq_reuse, on="request_id", how="left")
trace_df = trace_df.merge(batch_reuse, on="request_id", how="left")

# Estimated token-level reused/new KV blocks.
# This is a block-size estimate, not exact token count, unless each hash_id maps exactly to a full block.
for policy in ["sequential", "batch"]:
    trace_df[f"{policy}_reused_tokens_est"] = trace_df[f"{policy}_history_hit_block_refs"] * BLOCK_SIZE_TOKENS
    trace_df[f"{policy}_new_tokens_est"] = trace_df[f"{policy}_history_miss_block_refs"] * BLOCK_SIZE_TOKENS

reuse_summary = []
for policy in ["sequential", "batch"]:
    hit_col = f"{policy}_history_hit_block_refs"
    miss_col = f"{policy}_history_miss_block_refs"
    total_refs = trace_df[hit_col].sum() + trace_df[miss_col].sum()
    reuse_summary.append({
        "policy": policy,
        "total_block_refs": total_refs,
        "history_hit_block_refs": trace_df[hit_col].sum(),
        "history_miss_block_refs": trace_df[miss_col].sum(),
        "overall_history_hit_rate_refs": trace_df[hit_col].sum() / total_refs if total_refs else np.nan,
        "p50_request_hit_rate_refs": trace_df[f"{policy}_history_hit_rate_refs"].quantile(0.50),
        "p99_request_hit_rate_refs": trace_df[f"{policy}_history_hit_rate_refs"].quantile(0.99),
    })

reuse_summary_df = pd.DataFrame(reuse_summary)
display(reuse_summary_df)
display(trace_df[["request_id", "timestamp", "num_hash_blocks", "sequential_history_hit_rate_refs", "batch_history_hit_rate_refs"]].head(20))

,policy,total_block_refs,history_hit_block_refs,history_miss_block_refs,overall_history_hit_rate_refs,p50_request_hit_rate_refs,p99_request_hit_rate_refs
0,sequential,6293976,3738671,2555305,0.594008,0.681159,1.0
1,batch,6293976,3738668,2555308,0.594007,0.681159,1.0


,request_id,timestamp,num_hash_blocks,sequential_history_hit_rate_refs,batch_history_hit_rate_refs
0,0,0.000,50,0.000000,0.000000
1,1,0.206,24,0.000000,0.000000
2,2,0.223,1047,0.042980,0.042980
3,3,0.718,23,0.869565,0.869565
4,4,1.073,48,0.937500,0.937500
5,5,1.191,51,0.882353,0.882353
6,6,1.506,47,0.957447,0.957447
7,7,1.906,189,0.000000,0.000000
8,8,2.603,31,0.000000,0.000000
9,9,3.297,26,0.115385,0.115385


## 5. Prefix-sharing statistics

KV reuse is often prefix-driven. These metrics show whether requests share long prefixes with earlier requests.

In [7]:
def longest_common_prefix_len(a: list[Any], b: list[Any]) -> int:
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def longest_prefix_with_any_previous(sequences: Iterable[list[Any]]) -> list[int]:
    """
    Trie-based longest prefix shared with any previous request.
    Scales much better than pairwise O(N^2) comparison.
    """
    root = {}
    out = []

    for seq in sequences:
        node = root
        depth = 0
        for b in seq:
            if b in node:
                node = node[b]
                depth += 1
            else:
                break
        out.append(depth)

        # Insert sequence after measuring, so the current request does not match itself.
        node = root
        for b in seq:
            node = node.setdefault(b, {})

    return out


trace_df["lcp_with_previous_request"] = [
    np.nan if i == 0 else longest_common_prefix_len(trace_df.loc[i - 1, "hash_ids"], trace_df.loc[i, "hash_ids"])
    for i in range(len(trace_df))
]
trace_df["lcp_with_any_previous_request"] = longest_prefix_with_any_previous(trace_df["hash_ids"].tolist())
trace_df["lcp_any_previous_ratio"] = np.where(
    trace_df["num_hash_blocks"] > 0,
    trace_df["lcp_with_any_previous_request"] / trace_df["num_hash_blocks"],
    np.nan,
)

prefix_stats = percentile_table(
    trace_df,
    ["lcp_with_previous_request", "lcp_with_any_previous_request", "lcp_any_previous_ratio"],
)
display(prefix_stats)

,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,lcp_with_previous_request,43057,6.287317,14.611646,0.0,432.0,0.0,0.000000,0.000000,0.000000,0.000000,3.000000,45.000000,45.00000,45.0,432.0
1,lcp_with_any_previous_request,43058,84.480840,166.169930,0.0,5381.0,0.0,3.000000,3.000000,4.000000,45.000000,75.000000,230.000000,394.00000,746.0,5381.0
2,lcp_any_previous_ratio,43058,0.587164,0.318024,0.0,1.0,0.0,0.005267,0.017094,0.333333,0.632653,0.880622,0.957447,0.97619,1.0,1.0


In [8]:
# Top K-prefix groups help identify repeated templates/system prompts.
# For privacy-sensitive traces, do not print full long prefixes; print counts and prefix length only.

for k in [1, 2, 4, 8, 16, 32]:
    col = f"prefix_{k}"
    trace_df[col] = trace_df["hash_ids"].apply(lambda ids, k=k: tuple(ids[:k]))
    top = (
        trace_df.groupby(col, dropna=False)
        .agg(
            n_requests=("request_id", "count"),
            avg_input_tokens=("input_tokens", "mean"),
            avg_output_tokens=("output_tokens", "mean"),
        )
        .sort_values("n_requests", ascending=False)
        .head(10)
        .reset_index()
    )
    top["prefix_len"] = top[col].apply(len)
    print(f"\nTop prefix groups for k={k}")
    display(top[["prefix_len", "n_requests", "avg_input_tokens", "avg_output_tokens", col]])


Top prefix groups for k=1


,prefix_len,n_requests,avg_input_tokens,avg_output_tokens,prefix_1
0,1,21877,2206.344197,445.484481,"(1089,)"
1,1,14664,2886.751364,457.780210,"(0,)"
2,1,1655,481.739577,128.962538,"(50,)"
3,1,1161,3435.993971,979.389320,"(5252,)"
4,1,679,963.559647,121.300442,"(2483,)"
5,1,436,940.926606,211.931193,"(1275,)"
6,1,269,1608.293680,292.394052,"(7498,)"
7,1,268,1585.324627,135.257463,"(17965,)"
8,1,225,1558.666667,135.666667,"(8380,)"
9,1,220,260.913636,20.227273,"(78649,)"



Top prefix groups for k=2


,prefix_len,n_requests,avg_input_tokens,avg_output_tokens,prefix_2
0,2,21877,2206.344197,445.484481,"(1089, 1090)"
1,2,14664,2886.751364,457.780210,"(0, 1)"
2,2,1655,481.739577,128.962538,"(50, 51)"
3,2,1161,3435.993971,979.389320,"(5252, 5253)"
4,2,679,963.559647,121.300442,"(2483, 2484)"
5,2,436,940.926606,211.931193,"(1275, 1276)"
6,2,269,1608.293680,292.394052,"(7498, 7326)"
7,2,225,1558.666667,135.666667,"(8380, 8381)"
8,2,220,260.913636,20.227273,"(78649, 78650)"
9,2,217,911.036866,73.115207,"(8607, 8608)"



Top prefix groups for k=4


...

,prefix_len,n_requests,avg_input_tokens,avg_output_tokens,prefix_8
0,8,14664,2886.751364,457.780210,"(0, 1, 2, 3, 4, 5, 6, 7)"
1,8,1353,390.088692,87.274945,"(50, 51, 52, 53, 54, 55, 56, 57)"
2,8,1161,3435.993971,979.389320,"(5252, 5253, 5254, 5255, 5256, 5257, 5258, 5259)"
3,8,679,963.559647,121.300442,"(2483, 2484, 2485, 2486, 2487, 2488, 2489, 2490)"
4,8,436,940.926606,211.931193,"(1275, 1276, 1277, 1278, 1279, 1280, 1281, 1282)"
5,8,225,1558.666667,135.666667,"(8380, 8381, 8382, 8383, 8384, 8385, 8386, 8387)"
6,8,220,260.913636,20.227273,"(78649, 78650, 78651, 78652, 78653, 78654, 786..."
7,8,168,1090.797619,365.214286,"(1089, 1090, 1091, 14420, 14421, 14422, 14423,..."
8,8,129,258.124031,117.341085,"(23222, 23223, 23224, 23225, 23226, 23227, 232..."
9,8,127,2744.850394,362.401575,"(6439, 6440, 6441, 6442, 6443, 6444, 6445, 6446)"



Top prefix groups for k=16


,prefix_len,n_requests,avg_input_tokens,avg_output_tokens,prefix_16
0,16,14664,2886.751364,457.780210,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,16,1353,390.088692,87.274945,"(50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6..."
2,16,1161,3435.993971,979.389320,"(5252, 5253, 5254, 5255, 5256, 5257, 5258, 525..."
3,16,225,1558.666667,135.666667,"(8380, 8381, 8382, 8383, 8384, 8385, 8386, 838..."
4,16,168,1090.797619,365.214286,"(1089, 1090, 1091, 14420, 14421, 14422, 14423,..."
5,16,127,2744.850394,362.401575,"(6439, 6440, 6441, 6442, 6443, 6444, 6445, 644..."
6,16,119,1778.739496,349.109244,"(26917, 26918, 26919, 26920, 26921, 26922, 269..."
7,16,111,962.135135,115.342342,"(2483, 2484, 2485, 2486, 2487, 2488, 2489, 249..."
8,16,103,989.320388,110.834951,"(2483, 2484, 2485, 2486, 2487, 2488, 2489, 249..."
9,16,101,347.356436,350.673267,"(1089, 1090, 1091, 1306, 1307, 1308, 6830, 683..."



Top prefix groups for k=32


,prefix_len,n_requests,avg_input_tokens,avg_output_tokens,prefix_32
0,32,14664,2886.751364,457.780210,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,32,1161,3435.993971,979.389320,"(5252, 5253, 5254, 5255, 5256, 5257, 5258, 525..."
2,32,225,1558.666667,135.666667,"(8380, 8381, 8382, 8383, 8384, 8385, 8386, 838..."
3,23,50,368.000000,64.780000,"(50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6..."
4,32,36,6901.000000,208.250000,"(1089, 1090, 1091, 2068972, 2068973, 2068974, ..."
5,32,32,3228.687500,179.281250,"(17965, 1032545, 1032546, 1032547, 10504, 1050..."
6,32,28,4092.571429,141.821429,"(1506692, 1506693, 1506694, 1506695, 1506696, ..."
7,32,24,1657.125000,81.458333,"(7498, 7326, 171538, 171539, 171540, 171541, 1..."
8,32,23,7582.826087,155.608696,"(218300, 218301, 218302, 218303, 218304, 21830..."
9,32,20,13596.450000,445.450000,"(1089, 1090, 1091, 1947860, 1947861, 1947862, ..."


## 6. Block lifespan and reuse-distance analysis

A block's lifespan is the interval between first and last reference. Reuse distance is the gap between consecutive references to the same block.

In [9]:
def build_block_reference_table(df: pd.DataFrame, ids_col: str = "hash_ids") -> pd.DataFrame:
    """Explode request-level trace into one row per block reference."""
    refs = df[["request_id", "timestamp", "time_for_plot", ids_col]].explode(ids_col)
    refs = refs.rename(columns={ids_col: "block_id"})
    refs = refs.dropna(subset=["block_id"]).reset_index(drop=True)
    refs["ref_id"] = np.arange(len(refs))
    return refs


block_refs = build_block_reference_table(trace_df)

# For most cache analyses, one reference per block per request is enough.
# This prevents duplicate IDs within a single request from distorting lifespans.
block_req_refs = block_refs.drop_duplicates(["request_id", "block_id"]).copy()

block_stats = (
    block_req_refs.groupby("block_id", as_index=False)
    .agg(
        ref_requests=("request_id", "count"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        first_request_id=("request_id", "min"),
        last_request_id=("request_id", "max"),
    )
)
block_stats["reuse_count"] = block_stats["ref_requests"] - 1
block_stats["lifespan_time_raw"] = block_stats["last_timestamp"] - block_stats["first_timestamp"]
block_stats["lifespan_requests"] = block_stats["last_request_id"] - block_stats["first_request_id"] + 1
block_stats["is_singleton"] = block_stats["ref_requests"] == 1

# Reuse distance between consecutive references of the same block.
block_req_refs = block_req_refs.sort_values(["block_id", "request_id"])
block_req_refs["prev_request_id"] = block_req_refs.groupby("block_id")["request_id"].shift(1)
block_req_refs["prev_timestamp"] = block_req_refs.groupby("block_id")["timestamp"].shift(1)

reuse_distance_df = block_req_refs.dropna(subset=["prev_request_id"]).copy()
reuse_distance_df["reuse_distance_requests"] = reuse_distance_df["request_id"] - reuse_distance_df["prev_request_id"]
reuse_distance_df["reuse_distance_time_raw"] = reuse_distance_df["timestamp"] - reuse_distance_df["prev_timestamp"]

print(f"Unique blocks: {len(block_stats):,}")
print(f"Total block references: {len(block_refs):,}")
print(f"Singleton blocks: {block_stats['is_singleton'].sum():,} ({block_stats['is_singleton'].mean():.2%})")

block_life_stats = percentile_table(block_stats, ["ref_requests", "reuse_count", "lifespan_time_raw", "lifespan_requests"])
display(block_life_stats)

if len(reuse_distance_df) > 0:
    reuse_distance_stats = percentile_table(reuse_distance_df, ["reuse_distance_requests", "reuse_distance_time_raw"])
    display(reuse_distance_stats)
else:
    print("No repeated block references found after filtering; reuse distance table is empty.")

Unique blocks: 2,530,337
Total block references: 6,293,976
Singleton blocks: 1,355,641 (53.58%)


,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,ref_requests,2530337,2.445504,67.116360,1.0,21972.000,1.0,1.0,1.0,1.0,1.0,2.000,4.000,5.000,9.000,21972.000
1,reuse_count,2530337,1.445504,67.116360,0.0,21971.000,0.0,0.0,0.0,0.0,0.0,1.000,3.000,4.000,8.000,21971.000
2,lifespan_time_raw,2530337,334.427254,722.719035,0.0,7199.464,0.0,0.0,0.0,0.0,0.0,301.213,1114.785,1812.846,3291.198,7199.464
3,lifespan_requests,2530337,2157.544686,4653.771716,1.0,43058.000,1.0,1.0,1.0,1.0,1.0,1915.000,7211.000,12029.000,21757.000,43058.000


,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,reuse_distance_requests,3657613,1491.897806,2816.694631,1.0,42282.000,1.0,1.00,1.000,70.000,529.000,1512.000,3915.000,6500.000,14199.00,42282.000
1,reuse_distance_time_raw,3657613,231.356804,435.769104,0.0,7014.874,0.0,0.02,0.124,11.717,84.785,234.584,604.377,1024.281,2163.61,7014.874


## 7. Working set, live set, and estimated KV memory

The live set here means blocks whose first and last reference interval covers a timestamp. This is a demand-side lifespan view, not an exact cache residency view under a real eviction policy.

In [10]:
def kv_bytes_per_block(block_size_tokens: int, cfg: dict[str, int]) -> int:
    return int(
        block_size_tokens
        * cfg["n_layers"]
        * 2
        * cfg["n_kv_heads"]
        * cfg["head_dim"]
        * cfg["dtype_bytes"]
    )


def human_bytes(n: float) -> str:
    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    n = float(n)
    for unit in units:
        if abs(n) < 1024.0:
            return f"{n:,.2f} {unit}"
        n /= 1024.0
    return f"{n:,.2f} EiB"


BYTES_PER_BLOCK = kv_bytes_per_block(BLOCK_SIZE_TOKENS, KV_MODEL_CONFIG)
print(f"Estimated bytes per KV block: {human_bytes(BYTES_PER_BLOCK)}")

# Live set over unique timestamps.
live_rows = []
for _, row in time_df.iterrows():
    ts = row["timestamp"]
    active_blocks = ((block_stats["first_timestamp"] <= ts) & (block_stats["last_timestamp"] >= ts)).sum()
    live_rows.append({
        "timestamp": ts,
        "time_for_plot": row["time_for_plot"],
        "active_blocks_lifespan_view": active_blocks,
        "active_kv_bytes_est": active_blocks * BYTES_PER_BLOCK,
        "active_kv_gib_est": active_blocks * BYTES_PER_BLOCK / (1024 ** 3),
    })

live_df = pd.DataFrame(live_rows)
display(live_df.head(20))
display(percentile_table(live_df, ["active_blocks_lifespan_view", "active_kv_gib_est"]))

Estimated bytes per KV block: 2.00 MiB


,timestamp,time_for_plot,active_blocks_lifespan_view,active_kv_bytes_est,active_kv_gib_est
0,0.000,0.000000,50,104857600,0.097656
1,0.206,0.000206,73,153092096,0.142578
2,0.223,0.000223,1071,2246049792,2.091797
3,0.718,0.000718,317,664797184,0.619141
4,1.073,0.001073,319,668991488,0.623047
5,1.191,0.001191,325,681574400,0.634766
6,1.506,0.001506,327,685768704,0.638672
7,1.906,0.001906,512,1073741824,1.000000
8,2.603,0.002603,542,1136656384,1.058594
9,3.297,0.003297,564,1182793728,1.101562


,metric,count,mean,std,min,max,p00,p01,p05,p25,p50,p75,p90,p95,p99,p100
0,active_blocks_lifespan_view,42919,126787.032130,54916.698363,50.000000,210082.000000,50.000000,12117.160000,35830.800000,90824.500000,113378.000000,182868.000000,203306.200000,204712.10000,207699.640000,210082.000000
1,active_kv_gib_est,42919,247.630922,107.259176,0.097656,410.316406,0.097656,23.666328,69.982031,177.391602,221.441406,357.164062,397.082422,399.82832,405.663359,410.316406


In [ ]:
def compute_working_set_by_request(df: pd.DataFrame, windows: list[int], ids_col: str = "hash_ids") -> pd.DataFrame:
    """
    Sliding-window working set size by request.
    For very large traces, this can be optimized with reference counts; this simple version is readable and reliable.
    """
    rows = []
    all_ids = df[ids_col].tolist()
    for i in range(len(df)):
        row = {"request_id": df.loc[i, "request_id"], "timestamp": df.loc[i, "timestamp"], "time_for_plot": df.loc[i, "time_for_plot"]}
        for w in windows:
            start = max(0, i - w + 1)
            ws = {b for ids in all_ids[start : i + 1] for b in ids}
            row[f"working_set_blocks_last_{w}_req"] = len(ws)
            row[f"working_set_gib_last_{w}_req_est"] = len(ws) * BYTES_PER_BLOCK / (1024 ** 3)
        rows.append(row)
    return pd.DataFrame(rows)


working_set_df = compute_working_set_by_request(trace_df, WORKING_SET_WINDOWS)
display(working_set_df.head(20))

working_set_cols = [c for c in working_set_df.columns if c.startswith("working_set_blocks_last_")]
display(percentile_table(working_set_df, working_set_cols))

## 8. Visualization helpers

These helpers create one chart per figure. That makes notebook output easy to read and easy to copy into reports.

In [ ]:
def plot_hist(series: pd.Series, title: str, xlabel: str, bins: int = 50):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        print(f"Skip empty histogram: {title}")
        return
    plt.figure(figsize=(8, 4))
    plt.hist(s, bins=min(bins, max(5, int(np.sqrt(len(s))))))
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("count")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_ecdf(series: pd.Series, title: str, xlabel: str):
    s = np.sort(pd.to_numeric(series, errors="coerce").dropna().to_numpy())
    if len(s) == 0:
        print(f"Skip empty ECDF: {title}")
        return
    y = np.arange(1, len(s) + 1) / len(s)
    plt.figure(figsize=(8, 4))
    plt.plot(s, y)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("cumulative probability")
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_line(x, y, title: str, xlabel: str, ylabel: str):
    plt.figure(figsize=(9, 4))
    plt.plot(x, y, marker="o")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_scatter(x, y, title: str, xlabel: str, ylabel: str):
    plt.figure(figsize=(7, 5))
    plt.scatter(x, y, alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Request-level distributions
plot_hist(trace_df["input_tokens"], "Input token distribution", "input tokens")
plot_hist(trace_df["output_tokens"], "Output token distribution", "output tokens")
plot_hist(trace_df["total_tokens"], "Total token distribution", "total tokens")
plot_ecdf(trace_df["input_tokens"], "Input token ECDF", "input tokens")
plot_ecdf(trace_df["output_tokens"], "Output token ECDF", "output tokens")
plot_ecdf(trace_df["num_hash_blocks"], "Hash block count per request ECDF", "hash blocks per request")

# Input vs output shape
plot_scatter(trace_df["input_tokens"], trace_df["output_tokens"], "Input vs output tokens", "input tokens", "output tokens")

In [ ]:
# Time/batch view
x_label = trace_df.attrs.get("time_label", "time")
plot_line(time_df["time_for_plot"], time_df["n_requests"], "Requests per timestamp", x_label, "requests")
plot_line(time_df["time_for_plot"], time_df["total_tokens"], "Total tokens per timestamp", x_label, "tokens")
plot_line(time_df["time_for_plot"], time_df["unique_hash_blocks"], "Unique KV blocks per timestamp", x_label, "unique blocks")

In [ ]:
# KV-aware reuse view
plot_ecdf(trace_df["batch_history_hit_rate_refs"], "Batch-policy history hit-rate ECDF", "hit rate")
plot_ecdf(trace_df["sequential_history_hit_rate_refs"], "Sequential-policy history hit-rate ECDF", "hit rate")
plot_ecdf(trace_df["lcp_with_any_previous_request"], "Longest shared prefix with any previous request ECDF", "prefix length in blocks")
plot_ecdf(trace_df["lcp_any_previous_ratio"], "Longest shared prefix ratio ECDF", "prefix ratio")

In [ ]:
# Block popularity / Zipf-like behavior
block_freq = block_stats.sort_values("ref_requests", ascending=False)["ref_requests"].reset_index(drop=True)
if len(block_freq) > 0:
    ranks = np.arange(1, len(block_freq) + 1)
    plt.figure(figsize=(8, 4))
    plt.loglog(ranks, block_freq)
    plt.title("Block popularity rank-frequency plot")
    plt.xlabel("block rank by reference count")
    plt.ylabel("reference count")
    plt.grid(True, alpha=0.3)
    plt.show()

plot_ecdf(block_stats["ref_requests"], "Block reference count ECDF", "references per block")
plot_ecdf(block_stats["lifespan_requests"], "Block lifespan ECDF", "lifespan in request span")
plot_ecdf(block_stats["lifespan_time_raw"], "Block lifespan ECDF", "lifespan in raw timestamp units")

if len(reuse_distance_df) > 0:
    plot_ecdf(reuse_distance_df["reuse_distance_requests"], "Reuse distance ECDF", "request gap between reuses")
    plot_ecdf(reuse_distance_df["reuse_distance_time_raw"], "Reuse time gap ECDF", "timestamp gap between reuses")

In [ ]:
# Live set and working set
plot_line(
    live_df["time_for_plot"],
    live_df["active_blocks_lifespan_view"],
    "Active KV blocks over time, lifespan view",
    x_label,
    "active blocks",
)
plot_line(
    live_df["time_for_plot"],
    live_df["active_kv_gib_est"],
    "Estimated active KV memory over time, lifespan view",
    x_label,
    "GiB",
)

for w in WORKING_SET_WINDOWS:
    col = f"working_set_blocks_last_{w}_req"
    if col in working_set_df.columns:
        plot_line(
            working_set_df["request_id"],
            working_set_df[col],
            f"Working set over last {w} requests",
            "request id",
            "unique blocks",
        )

## 9. LRU cache simulation

This estimates how block-level hit rate changes with cache capacity. Capacity is in KV block count.

The simulation below is sequential. If your trace timestamps represent truly simultaneous batches, interpret this as an optimistic upper bound for same-timestamp reuse.

In [ ]:
def simulate_lru_cache(df: pd.DataFrame, capacity_blocks: int, ids_col: str = "hash_ids") -> dict[str, Any]:
    """Simple block-level LRU simulation."""
    if capacity_blocks <= 0:
        raise ValueError("capacity_blocks must be positive")

    cache = OrderedDict()
    total_hits = 0
    total_misses = 0
    per_request_hit_rates = []

    for _, row in df.iterrows():
        ids = list(row[ids_col])
        req_hits = 0
        req_misses = 0

        for b in ids:
            if b in cache:
                req_hits += 1
                cache.move_to_end(b)
            else:
                req_misses += 1
                cache[b] = None
                cache.move_to_end(b)
                while len(cache) > capacity_blocks:
                    cache.popitem(last=False)

        total_hits += req_hits
        total_misses += req_misses
        per_request_hit_rates.append(req_hits / len(ids) if ids else np.nan)

    total_refs = total_hits + total_misses
    return {
        "capacity_blocks": capacity_blocks,
        "capacity_gib_est": capacity_blocks * BYTES_PER_BLOCK / (1024 ** 3),
        "total_refs": total_refs,
        "hits": total_hits,
        "misses": total_misses,
        "overall_hit_rate": total_hits / total_refs if total_refs else np.nan,
        "p50_request_hit_rate": pd.Series(per_request_hit_rates).quantile(0.50),
        "p99_request_hit_rate": pd.Series(per_request_hit_rates).quantile(0.99),
    }


lru_results_df = pd.DataFrame([simulate_lru_cache(trace_df, c) for c in LRU_CAPACITIES])
display(lru_results_df)

plt.figure(figsize=(8, 4))
plt.plot(lru_results_df["capacity_blocks"], lru_results_df["overall_hit_rate"], marker="o")
plt.xscale("log")
plt.title("LRU cache hit-rate curve")
plt.xlabel("capacity in KV blocks, log scale")
plt.ylabel("overall block hit rate")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(lru_results_df["capacity_gib_est"], lru_results_df["overall_hit_rate"], marker="o")
plt.title("LRU cache hit-rate curve by estimated memory")
plt.xlabel("estimated KV capacity (GiB)")
plt.ylabel("overall block hit rate")
plt.grid(True, alpha=0.3)
plt.show()

## 10. Final report tables and export

These compact tables are usually enough for a first workload characterization report.

In [ ]:
# Compact report table.
report = {
    "n_requests": len(trace_df),
    "n_timestamps": trace_df["timestamp"].nunique(),
    "total_input_tokens": trace_df["input_tokens"].sum(),
    "total_output_tokens": trace_df["output_tokens"].sum(),
    "total_tokens": trace_df["total_tokens"].sum(),
    "total_hash_block_refs": trace_df["num_hash_blocks"].sum(),
    "unique_hash_blocks": block_stats["block_id"].nunique(),
    "global_block_reuse_factor_refs_per_unique_block": trace_df["num_hash_blocks"].sum() / max(block_stats["block_id"].nunique(), 1),
    "batch_overall_history_hit_rate_refs": reuse_summary_df.loc[reuse_summary_df["policy"] == "batch", "overall_history_hit_rate_refs"].iloc[0],
    "sequential_overall_history_hit_rate_refs": reuse_summary_df.loc[reuse_summary_df["policy"] == "sequential", "overall_history_hit_rate_refs"].iloc[0],
    "singleton_block_ratio": block_stats["is_singleton"].mean(),
    "bytes_per_block_est": BYTES_PER_BLOCK,
    "bytes_per_block_est_human": human_bytes(BYTES_PER_BLOCK),
}

report_df = pd.DataFrame([report]).T.reset_index()
report_df.columns = ["metric", "value"]
display(report_df)

print("Request p50/p99 quick view")
display(request_stats[["metric", "p50", "p90", "p95", "p99", "max"]])

print("Block lifespan p50/p99 quick view")
display(block_life_stats[["metric", "p50", "p90", "p95", "p99", "max"]])

In [ ]:
# Optional exports for later reporting.
OUTPUT_DIR = Path("kvcache_analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

trace_df.to_csv(OUTPUT_DIR / "request_level_features.csv", index=False)
time_df.to_csv(OUTPUT_DIR / "timestamp_level_features.csv", index=False)
block_stats.to_csv(OUTPUT_DIR / "block_lifespan_stats.csv", index=False)
reuse_distance_df.to_csv(OUTPUT_DIR / "block_reuse_distance.csv", index=False)
lru_results_df.to_csv(OUTPUT_DIR / "lru_capacity_curve.csv", index=False)
report_df.to_csv(OUTPUT_DIR / "summary_report.csv", index=False)

print(f"Exported CSV files to: {OUTPUT_DIR.resolve()}")